# **Download WSI**

In [ ]:
import argparse
import errno
import hashlib
import json
import re
import shutil
import struct
import threading
import time
import os
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

import numpy as np
import pandas as pd
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

# ---------------- KONFIGURASI ----------------
BASE_DIR = Path(os.environ.get("TCGA_BASE", "/workspace/TCGA-LUAD"))
XLSX_PATH = BASE_DIR / "EGFR.xlsx"
GDC_FILES = "https://api.gdc.cancer.gov/files"
GDC_DATA = "https://api.gdc.cancer.gov/data/"
PROJECT = "TCGA-LUAD"
TARGET_MAG = 40

N_MUTANT, N_WILDTYPE = 50, 150
MUTANT_SOURCE = "detail"          # "detail" (70 pasien) | "list" (84 pasien)
SELECT_MODE = "largest"           # "largest" | "random"
SEED = 42
ONE_SLIDE_PER_PATIENT = True

MAX_WORKERS = 4
MAX_RETRIES = 5
BACKOFF_BASE = 5
RETRY_FAILED_ON_RESUME = True

WSI_DIR = BASE_DIR / "WSI"
PART_DIR = BASE_DIR / "parts"
GDC_CSV = BASE_DIR / "gdc_check_output" / "gdc_dx_slides_all.csv"
MAG_CACHE = BASE_DIR / "gdc_check_output" / "magnification_cache.csv"
PROGRESS_CSV = WSI_DIR / "download_progress.csv"
LABELS_CSV = WSI_DIR / "labels.csv"
REPORT_CSV = WSI_DIR / "selection_report.csv"

TCGA_RE = re.compile(r"TCGA-[A-Z0-9]{2}-[A-Z0-9]{4}", re.I)
FN_RE = re.compile(r"^(TCGA-[A-Z0-9]{2}-[A-Z0-9]{4})-(\d{2})[A-Z]-[A-Z0-9]{2}-(DX\d+)", re.I)

_lock = threading.Lock()
plan = None
mag_cache = {}


class NonRetryable(Exception):
    pass


class DiskFull(Exception):
    pass


def log(msg):
    with _lock:
        print(msg, flush=True)


# ---------------- LABEL DARI EXCEL ----------------
def ids_from(series):
    return {m.upper() for v in series.dropna().astype(str) for m in TCGA_RE.findall(v)}


def read_label_sets(xlsx):
    if not Path(xlsx).exists():
        raise SystemExit(f"File tidak ditemukan: {xlsx}")
    xl = pd.ExcelFile(xlsx)
    names = {n.lower().strip(): n for n in xl.sheet_names}
    sh_wt = next((n for k, n in names.items() if "wild" in k), None)
    sh_detail = next((n for k, n in names.items() if "egfr" in k and "mut" in k), None)
    sh_list = names.get("mutation")
    if not sh_wt or not sh_detail:
        raise SystemExit(f"Sheet 'Wild Type' / 'EGFR Mutation' tidak ditemukan. Sheet ada: {xl.sheet_names}")

    wt = ids_from(xl.parse(sh_wt, header=None).iloc[:, 0])          # header ikut dibaca (baris pertama = ID)
    detail_df = xl.parse(sh_detail)
    if "Sample ID" not in detail_df.columns:
        raise SystemExit("Kolom 'Sample ID' tidak ada di sheet detail mutasi.")
    detail = ids_from(detail_df["Sample ID"])
    lst = ids_from(xl.parse(sh_list, header=None).iloc[:, 0]) if sh_list else set()
    return {"wildtype": wt, "detail": detail, "list": lst or detail}


def print_label_summary(S):
    wt, d, l = S["wildtype"], S["detail"], S["list"]
    print("=" * 66)
    print("LABEL DARI EXCEL (level pasien)")
    print("=" * 66)
    print(f"Mutant  - sheet detail : {len(d)} pasien")
    print(f"Mutant  - sheet list   : {len(l)} pasien (di luar detail: {len(l - d)})")
    print(f"Wildtype terverifikasi : {len(wt)} pasien")
    print(f"Irisan wildtype & mutant: {len(wt & (d | l))} (harus 0)")
    if wt & (d | l):
        raise SystemExit("Pasien muncul di wildtype DAN mutant. Periksa Excel.")
    if l - d:
        print(f"14-an 'Mutation tanpa detail' = {sorted(l - d)}")


# ---------------- AUDIT labels.csv LAMA ----------------
def audit(S, labels_csv):
    if not labels_csv.exists():
        raise SystemExit(f"{labels_csv} tidak ada.")
    lab = pd.read_csv(labels_csv)
    lab["pid"] = lab["patient_id"].astype(str).str.upper().str[:12]
    is_mut = lab["label"].astype(str).str.lower().str.startswith("mutant")
    wt, d, l = S["wildtype"], S["detail"], S["list"]
    print_label_summary(S)
    print("\n" + "=" * 66)
    print(f"AUDIT {labels_csv}  ({len(lab)} slide)")
    print("=" * 66)
    m, w = lab[is_mut], lab[~is_mut]
    print(f"Berlabel Mutant   : {len(m)} | ada di detail: {m['pid'].isin(d).sum()} | "
          f"hanya di list: {(m['pid'].isin(l) & ~m['pid'].isin(d)).sum()} | di luar Excel: {(~m['pid'].isin(l | d)).sum()}")
    bad_list = w[w["pid"].isin(l)]
    bad_none = w[~w["pid"].isin(wt) & ~w["pid"].isin(l)]
    print(f"Berlabel Wildtype : {len(w)} | terverifikasi (sheet Wild Type): {w['pid'].isin(wt).sum()}")
    print(f"   !! bermutasi EGFR menurut sheet 'Mutation' (SALAH LABEL): {len(bad_list)} {sorted(bad_list['pid'])}")
    print(f"   !  tidak ada di sheet mana pun (tak terverifikasi)       : {len(bad_none)} {sorted(bad_none['pid'])[:20]}")
    if len(bad_list) == 0 and len(bad_none) == 0:
        print("\nLabel lama BERSIH: semua wildtype terverifikasi.")
    else:
        print("\nAda label wildtype yang tidak terverifikasi. Jalankan ulang seleksi dengan skrip ini.")


# ---------------- DAFTAR SLIDE DX DARI GDC ----------------
def make_session():
    s = requests.Session()
    retry = Retry(total=3, backoff_factor=1.5, status_forcelist=(429, 500, 502, 503, 504), allowed_methods=("GET",))
    ad = HTTPAdapter(max_retries=retry, pool_maxsize=max(MAX_WORKERS, 4))
    s.mount("https://", ad)
    s.mount("http://", ad)
    return s


def fetch_gdc_dx(session):
    def query(extra):
        content = [{"op": "=", "content": {"field": "cases.project.project_id", "value": PROJECT}},
                   {"op": "=", "content": {"field": "data_type", "value": "Slide Image"}},
                   {"op": "=", "content": {"field": "access", "value": "open"}}] + extra
        params = {"filters": json.dumps({"op": "and", "content": content}),
                  "fields": "file_id,file_name,file_size,md5sum,cases.submitter_id",
                  "format": "JSON", "size": "3000"}
        r = session.get(GDC_FILES, params=params, timeout=90)
        r.raise_for_status()
        return r.json()["data"]["hits"]

    hits = query([{"op": "=", "content": {"field": "experimental_strategy", "value": "Diagnostic Slide"}}])
    if not hits:
        hits = query([])
    rows = []
    for h in hits:
        m = FN_RE.match(h["file_name"])
        if not m:                                   # bukan file DX (mis. tissue slide TS/BS)
            continue
        rows.append({"file_id": h["file_id"], "file_name": h["file_name"], "file_size": int(h["file_size"]),
                     "md5sum": h["md5sum"], "patient_id": m.group(1).upper(),
                     "sample_code": m.group(2), "dx": m.group(3).upper()})
    return pd.DataFrame(rows)


def load_dx_table(session, refresh):
    if GDC_CSV.exists() and not refresh:
        df = pd.read_csv(GDC_CSV)
        print(f"Daftar slide DX dari cache: {GDC_CSV} ({len(df)} file)")
        return df
    print("Mengambil daftar slide DX dari GDC API ...")
    try:
        df = fetch_gdc_dx(session)
    except Exception as e:
        raise SystemExit(f"Gagal mengakses GDC API: {repr(e)}\nPastikan Pod punya internet, lalu ulangi.")
    if df.empty:
        raise SystemExit("GDC tidak mengembalikan slide DX. Cek filter project/data_type.")
    GDC_CSV.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(GDC_CSV, index=False)
    print(f"Slide DX: {len(df)} file | {df['patient_id'].nunique()} pasien | disimpan: {GDC_CSV}")
    return df


# ---------------- RENCANA SELEKSI ----------------
def build_plan(dx, S):
    mutant = S[MUTANT_SOURCE]
    wt = S["wildtype"]
    df = dx[dx["sample_code"].astype(str).str.zfill(2) == "01"].copy()          # primary tumor
    df["label"] = np.where(df["patient_id"].isin(mutant), "Mutant",
                           np.where(df["patient_id"].isin(wt), "Wildtype", ""))
    df = df[df["label"] != ""].copy()
    df["label_source"] = np.where(df["label"] == "Mutant",
                                  f"excel_mutant_{MUTANT_SOURCE}", "excel_wild_type_verified")
    df["size_mb"] = df["file_size"] / 1e6

    parts = []
    rng = np.random.default_rng(SEED)
    for lab in ("Mutant", "Wildtype"):
        d = df[df["label"] == lab]
        if SELECT_MODE == "largest":
            d = d.sort_values(["file_size", "file_name"], ascending=[False, True])
            if ONE_SLIDE_PER_PATIENT:
                d = d.drop_duplicates("patient_id")
        else:
            d = d.sort_values(["patient_id", "dx", "file_name"])
            if ONE_SLIDE_PER_PATIENT:
                d = d.drop_duplicates("patient_id")
            d = d.iloc[rng.permutation(len(d))]
        d = d.copy()
        d["rank"] = range(1, len(d) + 1)
        parts.append(d)
    out = pd.concat(parts).set_index("file_id")
    out["status"] = "candidate"
    out["mag"] = np.nan
    out["attempts"] = 0
    out["error"] = ""
    out["finished_at"] = ""
    return out


def load_progress(p):
    if PROGRESS_CSV.exists():
        old = pd.read_csv(PROGRESS_CSV).set_index("file_id")
        common = p.index.intersection(old.index)
        for c in ("status", "error", "finished_at"):
            p.loc[common, c] = old.loc[common, c].fillna("").astype(object)
        p.loc[common, "attempts"] = old.loc[common, "attempts"].fillna(0).astype(int)
        p.loc[common, "mag"] = old.loc[common, "mag"]
    p.loc[p["status"] == "pending", "status"] = "candidate"
    if RETRY_FAILED_ON_RESUME:
        p.loc[p["status"] == "failed", "status"] = "candidate"
    p.loc[(p["status"] == "skip_mag") & p["mag"].isna(), "status"] = "candidate"   # gagal baca header -> coba lagi
    return p


def save_progress():
    tmp = PROGRESS_CSV.with_suffix(".csv.tmp")
    plan.to_csv(tmp)
    os.replace(tmp, PROGRESS_CSV)


def set_state(fid, status, attempts=None, error=""):
    with _lock:
        plan.at[fid, "status"] = status
        if attempts is not None:
            plan.at[fid, "attempts"] = attempts
        plan.at[fid, "error"] = error[:300]
        plan.at[fid, "finished_at"] = time.strftime("%Y-%m-%d %H:%M:%S") if status == "done" else ""
        save_progress()


# ---------------- MAGNIFIKASI (header SVS lewat HTTP Range) ----------------
class RemoteReader:
    def __init__(self, session, fid):
        self.s, self.fid = session, fid

    def read(self, offset, length):
        end = offset + length - 1
        r = self.s.get(GDC_DATA + self.fid, headers={"Range": f"bytes={offset}-{end}"},
                       stream=True, timeout=(30, 60))
        try:
            if r.status_code == 206:
                return r.content
            r.raise_for_status()
            if end > 32 * 1024 * 1024:
                raise IOError("server mengabaikan Range dan offset terlalu jauh")
            buf = bytearray()
            for chunk in r.iter_content(1 << 16):
                buf += chunk
                if len(buf) > end:
                    break
            return bytes(buf[offset:offset + length])
        finally:
            r.close()


class LocalReader:
    def __init__(self, path):
        self.path = path

    def read(self, offset, length):
        with open(self.path, "rb") as f:
            f.seek(offset)
            return f.read(length)


def first_ifd_description(rd):
    h = rd.read(0, 16)
    if h[:2] == b"II":
        e = "<"
    elif h[:2] == b"MM":
        e = ">"
    else:
        raise ValueError(f"bukan TIFF (header={h[:8].hex()})")
    magic = struct.unpack(e + "H", h[2:4])[0]
    if magic == 42:
        off = struct.unpack(e + "I", h[4:8])[0]
        n = struct.unpack(e + "H", rd.read(off, 2))[0]
        esz, cnt_len, val_len, ptr_fmt, hdr = 12, 4, 4, "I", 2
    elif magic == 43:
        off = struct.unpack(e + "Q", h[8:16])[0]
        n = struct.unpack(e + "Q", rd.read(off, 8))[0]
        esz, cnt_len, val_len, ptr_fmt, hdr = 20, 8, 8, "Q", 8
    else:
        raise ValueError(f"magic TIFF tidak dikenal: {magic}")
    if not 0 < n < 1000:
        raise ValueError(f"jumlah entri IFD tidak masuk akal: {n}")
    raw = rd.read(off + hdr, n * esz)
    for i in range(n):
        ent = raw[i * esz:(i + 1) * esz]
        if struct.unpack(e + "H", ent[0:2])[0] != 270:
            continue
        count = struct.unpack(e + ("I" if cnt_len == 4 else "Q"), ent[4:4 + cnt_len])[0]
        val = ent[4 + cnt_len:4 + cnt_len + val_len]
        data = val[:count] if count <= val_len else rd.read(
            struct.unpack(e + ptr_fmt, val)[0], min(count, 64 * 1024))
        return data.split(b"\x00")[0].decode("latin-1", errors="replace")
    raise ValueError("tag ImageDescription tidak ada")


def parse_mag(desc):
    m = re.search(r"AppMag\s*=\s*(\d+)", desc)
    if m:
        return int(m.group(1))
    m = re.search(r"MPP\s*=\s*([\d.]+)", desc)
    if m:
        return 40 if float(m.group(1)) < 0.35 else 20
    return None


def verify_svs(path):
    try:
        mag = parse_mag(first_ifd_description(LocalReader(path)))
    except Exception as e:
        return False, f"struktur SVS rusak: {e}"
    return (True, "") if mag == TARGET_MAG else (False, f"magnifikasi {mag} != {TARGET_MAG}")


def get_magnification(session, fid):
    """Magnifikasi dari header SVS jarak jauh (beberapa KB saja). None jika gagal dibaca."""
    if fid in mag_cache:
        return mag_cache[fid]
    for attempt in range(1, 4):
        try:
            mag = parse_mag(first_ifd_description(RemoteReader(session, fid)))
            if mag is not None:
                mag_cache[fid] = mag
                MAG_CACHE.parent.mkdir(parents=True, exist_ok=True)
                pd.DataFrame({"file_id": list(mag_cache), "magnification": list(mag_cache.values())}).to_csv(MAG_CACHE, index=False)
            return mag
        except Exception as e:
            if attempt == 3:
                log(f"   ! gagal membaca header {fid}: {type(e).__name__}: {e}")
            time.sleep(2 * attempt)
    return None


def file_md5(path, chunk=8 * 1024 * 1024):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(chunk), b""):
            h.update(b)
    return h.hexdigest()


# ---------------- KUOTA: pilih kandidat 40x berikutnya ----------------
def mark_existing_done():
    for fid, r in plan[plan["status"].isin(["candidate", "pending"])].iterrows():
        dest = WSI_DIR / r["file_name"]
        if dest.exists() and dest.stat().st_size == int(r["file_size"]):
            plan.at[fid, "status"] = "done"
            plan.at[fid, "mag"] = TARGET_MAG
            plan.at[fid, "finished_at"] = "sudah ada"


def ensure_quota(session, label, quota):
    """Tambah kandidat 'pending' (sudah terbukti 40x) sampai done+pending >= kuota."""
    while True:
        sub = plan[plan["label"] == label]
        if int(sub["status"].isin(["pending", "done"]).sum()) >= quota:
            return
        nxt = sub[sub["status"] == "candidate"].sort_values("rank").head(1)
        if nxt.empty:
            log(f"PERINGATAN: kandidat {label} 40x habis ({int(sub['status'].isin(['pending', 'done']).sum())}/{quota}).")
            return
        fid = nxt.index[0]
        mag = plan.at[fid, "mag"]
        if pd.isna(mag):
            mag = mag_cache.get(fid) or get_magnification(session, fid)
        else:
            mag = int(mag)
        plan.at[fid, "mag"] = np.nan if mag is None else mag
        if mag == TARGET_MAG:
            plan.at[fid, "status"] = "pending"
        else:
            plan.at[fid, "status"] = "skip_mag"
            plan.at[fid, "error"] = "magnifikasi tidak terbaca" if mag is None else f"magnifikasi {mag}x"
        time.sleep(0.1)


# ---------------- DOWNLOAD ----------------
def stream_to_part(session, fid, part, have):
    headers = {"Range": f"bytes={have}-"} if have else {}
    with session.get(GDC_DATA + fid, headers=headers, stream=True, timeout=(30, 120)) as r:
        if r.status_code == 206:
            mode = "ab"
        else:
            r.raise_for_status()
            mode = "wb"
        with open(part, mode) as f:
            for chunk in r.iter_content(1 << 20):
                f.write(chunk)


def download_one(session, fid):
    row = plan.loc[fid]
    expected, md5 = int(row["file_size"]), str(row["md5sum"]).lower()
    name, dest = row["file_name"], WSI_DIR / row["file_name"]
    part = PART_DIR / f"{fid}.part"
    err, t0 = "", time.time()

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            have = part.stat().st_size if part.exists() else 0
            if have > expected:
                part.unlink()
                have = 0
            if have < expected:
                if have:
                    log(f"   ~ lanjut {name} dari {have / 1e6:.0f} MB")
                stream_to_part(session, fid, part, have)
            size = part.stat().st_size
            if size != expected:
                raise IOError(f"ukuran belum lengkap ({size}/{expected})")
            if file_md5(part) != md5:
                part.unlink()
                raise ValueError("md5 tidak cocok (file korup)")
            ok, info = verify_svs(part)
            if not ok:
                part.unlink()
                raise NonRetryable(info)
            os.replace(part, dest)
            set_state(fid, "done", attempt)
            dt = max(time.time() - t0, 0.1)
            log(f"   OK  {name}  ({expected / 1e6:.0f} MB, {expected / 1e6 / dt:.1f} MB/s)")
            return
        except NonRetryable as e:
            err = str(e)
            break
        except OSError as e:
            if e.errno in (errno.ENOSPC, errno.EDQUOT):
                raise DiskFull(f"volume penuh saat menulis {name}")
            err = f"{type(e).__name__}: {e}"
            log(f"   !   {name} percobaan {attempt}/{MAX_RETRIES}: {err}")
        except (requests.RequestException, ValueError) as e:
            err = f"{type(e).__name__}: {e}"
            log(f"   !   {name} percobaan {attempt}/{MAX_RETRIES}: {err}")
        if attempt < MAX_RETRIES:
            time.sleep(BACKOFF_BASE * attempt)
    set_state(fid, "failed", MAX_RETRIES, err)
    log(f"   GAGAL {name}: {err}  -> diganti kandidat berikutnya")


def run_class(session, label, quota):
    while True:
        ensure_quota(session, label, quota)
        sub = plan[plan["label"] == label].sort_values("rank")
        need = quota - int((sub["status"] == "done").sum())
        if need <= 0:
            return
        todo = sub[sub["status"] == "pending"].head(need)
        if todo.empty:
            log(f"PERINGATAN: {label} terkumpul {quota - need}/{quota}; kandidat habis.")
            return
        log(f"\n[{label}] mengunduh {len(todo)} file (target {quota}, selesai {quota - need}) ...")
        ex = ThreadPoolExecutor(MAX_WORKERS)
        try:
            for f in as_completed([ex.submit(download_one, session, fid) for fid in todo.index]):
                f.result()
        except BaseException:
            ex.shutdown(wait=False, cancel_futures=True)
            raise
        ex.shutdown(wait=True)


# ---------------- OUTPUT ----------------
def selected_done():
    parts = []
    for lab, q in (("Mutant", N_MUTANT), ("Wildtype", N_WILDTYPE)):
        d = plan[(plan["label"] == lab) & (plan["status"] == "done")].sort_values("rank").head(q)
        parts.append(d)
    return pd.concat(parts)


def write_labels():
    d = selected_done().reset_index()
    d = d[["file_name", "patient_id", "label", "label_source", "size_mb"]].sort_values(["label", "file_name"])
    d.to_csv(LABELS_CSV, index=False)


def summary(title):
    print("\n" + "=" * 66)
    print(title)
    print("=" * 66)
    for lab, q in (("Mutant", N_MUTANT), ("Wildtype", N_WILDTYPE)):
        sub = plan[plan["label"] == lab]
        got = sub[sub["status"].isin(["pending", "done"])].sort_values("rank").head(q)
        done = int((got["status"] == "done").sum())
        print(f"{lab:9s}: terpilih {len(got):3d}/{q} | sudah ada/selesai {done:3d} | "
              f"{got['file_size'].sum() / 1e9:7.1f} GB | median {got['size_mb'].median():.0f} MB | "
              f"dilewati bukan 40x: {int((sub['status'] == 'skip_mag').sum())}")
    tot = pd.concat([plan[(plan['label'] == l) & plan['status'].isin(['pending', 'done'])].sort_values('rank').head(q)
                     for l, q in (("Mutant", N_MUTANT), ("Wildtype", N_WILDTYPE))])
    left = tot[tot["status"] != "done"]["file_size"].sum() / 1e9
    print(f"TOTAL terpilih: {len(tot)} slide | {tot['file_size'].sum() / 1e9:.1f} GB | masih perlu diunduh: {left:.1f} GB")


# ---------------- MAIN ----------------
def main():
    global plan, mag_cache, MUTANT_SOURCE, SELECT_MODE, SEED, N_MUTANT, N_WILDTYPE, MAX_WORKERS, XLSX_PATH
    ap = argparse.ArgumentParser()
    ap.add_argument("--xlsx", type=Path, default=XLSX_PATH)
    ap.add_argument("--mutant-source", choices=["detail", "list"], default=MUTANT_SOURCE)
    ap.add_argument("--select", choices=["largest", "random"], default=SELECT_MODE)
    ap.add_argument("--seed", type=int, default=SEED)
    ap.add_argument("--n-mutant", type=int, default=N_MUTANT)
    ap.add_argument("--n-wildtype", type=int, default=N_WILDTYPE)
    ap.add_argument("--workers", type=int, default=MAX_WORKERS)
    ap.add_argument("--refresh", action="store_true", help="ambil ulang daftar slide dari GDC")
    ap.add_argument("--audit", action="store_true", help="cek WSI/labels.csv lama terhadap Excel, lalu keluar")
    ap.add_argument("--dry-run", action="store_true", help="pilih slide + cek 40x, tanpa download")
    args = ap.parse_args()
    XLSX_PATH, MUTANT_SOURCE, SELECT_MODE, SEED = args.xlsx, args.mutant_source, args.select, args.seed
    N_MUTANT, N_WILDTYPE, MAX_WORKERS = args.n_mutant, args.n_wildtype, args.workers

    S = read_label_sets(XLSX_PATH)
    if args.audit:
        audit(S, LABELS_CSV)
        return
    print_label_summary(S)
    print(f"\nMutant dipakai: sheet '{MUTANT_SOURCE}' ({len(S[MUTANT_SOURCE])} pasien) | "
          f"seleksi: {SELECT_MODE} | target {N_MUTANT} mutant + {N_WILDTYPE} wildtype @ {TARGET_MAG}x")
    if MUTANT_SOURCE == "detail" and S["list"] - S["detail"]:
        print(f"{len(S['list'] - S['detail'])} pasien 'Mutation tanpa detail' tidak dipakai di kelas mana pun.")

    WSI_DIR.mkdir(parents=True, exist_ok=True)
    PART_DIR.mkdir(parents=True, exist_ok=True)
    session = make_session()

    dx = load_dx_table(session, args.refresh)
    if MAG_CACHE.exists():
        mc = pd.read_csv(MAG_CACHE)
        mag_cache = dict(zip(mc["file_id"], mc["magnification"].astype(int)))

    plan = load_progress(build_plan(dx, S))
    for lab, q in (("Mutant", N_MUTANT), ("Wildtype", N_WILDTYPE)):
        n = int((plan["label"] == lab).sum())
        print(f"Kandidat {lab:9s}: {n} pasien dengan slide DX primer" + ("  <-- KURANG dari target!" if n < q else ""))

    mark_existing_done()

    if args.dry_run:
        for lab, q in (("Mutant", N_MUTANT), ("Wildtype", N_WILDTYPE)):
            ensure_quota(session, lab, q)
        save_progress()
        plan.to_csv(REPORT_CSV)
        summary("DRY RUN (tidak ada yang diunduh)")
        print(f"Laporan seleksi: {REPORT_CSV}")
        return

    try:
        run_class(session, "Mutant", N_MUTANT)
        run_class(session, "Wildtype", N_WILDTYPE)
    except DiskFull as e:
        print(f"\nBERHENTI: {e}. Perbesar volume atau hapus file lain, lalu jalankan ulang (resume).")
    except KeyboardInterrupt:
        print("\nDihentikan manual. Jalankan ulang untuk melanjutkan.")
    finally:
        with _lock:
            save_progress()
            plan.to_csv(REPORT_CSV)
            write_labels()

    summary("LAPORAN AKHIR")
    failed = plan[plan["status"] == "failed"]
    if len(failed):
        print(f"\nGagal ({len(failed)}), akan dicoba lagi saat dijalankan ulang:")
        print(failed[["file_name", "error"]].to_string())
    used = set(selected_done()["file_name"])
    extra = sorted(p.name for p in WSI_DIR.glob("*.svs") if p.name not in used)
    if extra:
        print(f"\n{len(extra)} file .svs di WSI/ TIDAK termasuk seleksi akhir (tidak ada di labels.csv), contoh: {extra[:3]}")
        print("Tahap split/patch memakai labels.csv; tissue detection memproses semua .svs, jadi pindahkan/hapus bila tak dipakai.")
    print(f"\nLabel : {LABELS_CSV}\nLaporan: {REPORT_CSV}\nProgres: {PROGRESS_CSV}")


if __name__ == "__main__":
    main()


# **Split Dataset**

In [ ]:
import os
import shutil
from pathlib import Path

import numpy as np
import pandas as pd

# ---------------- KONFIGURASI ----------------
BASE_DIR = Path(os.environ.get("TCGA_BASE", "/workspace/TCGA-LUAD"))
WSI_DIR = BASE_DIR / "WSI"
LABELS_CSV = WSI_DIR / "labels.csv"          # dibuat oleh download_wsi_luad.py
SPLIT_CSV = WSI_DIR / "split.csv"

TRAIN_RATIO = 0.80
SEED = 42

# "csv"  : hanya tulis split.csv (disarankan, file .svs tidak dipindah)
# "move" : pindahkan .svs ke WSI/train/<label>/ dan WSI/test/<label>/
# "undo" : kembalikan semua .svs ke WSI/
MODE = "csv"


def make_split(df):
    """Split per pasien & stratified per label -> tidak ada kebocoran data pasien."""
    rng = np.random.default_rng(SEED)
    pat = df.groupby("patient_id")["label"].first().reset_index()

    train_ids = []
    for lab, g in pat.groupby("label"):
        ids = np.sort(g["patient_id"].to_numpy())     # urut dulu -> hasil deterministik
        rng.shuffle(ids)
        n = len(ids)
        n_train = int(round(n * TRAIN_RATIO))
        if n >= 2:
            n_train = min(max(n_train, 1), n - 1)
        train_ids.extend(ids[:n_train])

    out = df.copy()
    out["split"] = np.where(out["patient_id"].isin(set(train_ids)), "train", "test")
    return out


def check_split(out):
    tr = set(out.loc[out["split"] == "train", "patient_id"])
    te = set(out.loc[out["split"] == "test", "patient_id"])
    assert not (tr & te), "Ada pasien di train dan test sekaligus!"
    assert out["file_name"].is_unique, "Ada file_name duplikat"


def summarize(out):
    print("=" * 56)
    print("RINGKASAN SPLIT")
    print("=" * 56)
    tab = pd.crosstab(out["label"], out["split"]).reindex(columns=["train", "test"], fill_value=0)
    tab["total"] = tab.sum(axis=1)
    tab.loc["TOTAL"] = tab.sum()
    print(tab.to_string())
    for s in ("train", "test"):
        d = out[out["split"] == s]
        if len(d):
            print(f"  {s:5s}: {len(d)} slide | {(d['label'] == 'Mutant').mean() * 100:.1f}% Mutant")


def move_files(out):
    moved = 0
    for _, r in out.iterrows():
        src = WSI_DIR / r["file_name"]
        dst = WSI_DIR / r["split"] / r["label"] / r["file_name"]
        if dst.exists():
            continue
        if not src.exists():
            print(f"  ! tidak ditemukan: {src.name}")
            continue
        dst.parent.mkdir(parents=True, exist_ok=True)
        os.replace(src, dst)
        moved += 1
    print(f"\n{moved} file dipindahkan ke {WSI_DIR}/train dan {WSI_DIR}/test")


def undo_move():
    moved = 0
    for s in ("train", "test"):
        for p in (WSI_DIR / s).rglob("*.svs"):
            os.replace(p, WSI_DIR / p.name)
            moved += 1
    for s in ("train", "test"):
        shutil.rmtree(WSI_DIR / s, ignore_errors=True)
    print(f"{moved} file dikembalikan ke {WSI_DIR}")


def main():
    if MODE == "undo":
        undo_move()
        return
    if not LABELS_CSV.exists():
        raise FileNotFoundError(f"{LABELS_CSV} tidak ada. Jalankan download_wsi_luad.py dulu.")

    df = pd.read_csv(LABELS_CSV)
    if df.empty:
        raise SystemExit("labels.csv kosong, belum ada slide yang selesai diunduh.")

    def exists(name):
        return (WSI_DIR / name).exists() or any(WSI_DIR.glob(f"*/*/{name}"))
    ada = df["file_name"].map(exists)
    if (~ada).any():
        print(f"PERINGATAN: {(~ada).sum()} file di labels.csv tidak ada di disk, dilewati.")
    df = df[ada].copy()

    out = make_split(df)
    check_split(out)
    summarize(out)

    out = out.sort_values(["split", "label", "file_name"])
    tmp = SPLIT_CSV.with_suffix(".csv.tmp")
    out.to_csv(tmp, index=False)
    os.replace(tmp, SPLIT_CSV)
    print(f"\nSplit disimpan: {SPLIT_CSV}")

    if MODE == "move":
        move_files(out)
    else:
        print("MODE = 'csv': file .svs tidak dipindah. Baca split.csv di tahap berikutnya.")


if __name__ == "__main__":
    main()


# **Tissue Detection**

In [ ]:
import gc
import os
import time
from concurrent.futures import ProcessPoolExecutor, as_completed
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import openslide
from skimage.morphology import remove_small_objects, remove_small_holes

cv2.setNumThreads(1)          # 1 thread OpenCV per proses; paralelisme dari pool

# ---------------- LOKASI ----------------
BASE_DIR = Path(os.environ.get("TCGA_BASE", "/workspace/TCGA-LUAD"))
WSI_INPUT_DIR = BASE_DIR / "WSI"
OUTPUT_DIR = BASE_DIR / "Tissue_Detection"
SUMMARY_CSV = OUTPUT_DIR / "summary_tissue_detection.csv"

# ---------------- PARAMETER ----------------
MASK_DOWNSAMPLE = 32      # thumbnail = level 0 / 32
MIN_OBJECT_SIZE = 500
HOLE_AREA = 5000
CLOSE_KERNEL = 11
OPEN_KERNEL = 5
N_WORKERS = int(os.environ.get("N_WORKERS", min(os.cpu_count() or 4, 8)))


def clean_mask(mask):
    """Kompatibel dengan scikit-image lama dan baru."""
    try:
        mask = remove_small_objects(mask, max_size=MIN_OBJECT_SIZE)
        mask = remove_small_holes(mask, max_size=HOLE_AREA)
    except TypeError:
        mask = remove_small_objects(mask, min_size=MIN_OBJECT_SIZE)
        mask = remove_small_holes(mask, area_threshold=HOLE_AREA)
    return mask


def tissue_detection_otsu(slide_path, mask_downsample=32):
    slide = openslide.OpenSlide(str(slide_path))
    try:
        width, height = slide.dimensions
        thumb_w = max(1, int(width / mask_downsample))
        thumb_h = max(1, int(height / mask_downsample))
        thumbnail = np.array(slide.get_thumbnail((thumb_w, thumb_h)).convert("RGB"))

        hsv = cv2.cvtColor(thumbnail, cv2.COLOR_RGB2HSV)
        saturation = cv2.GaussianBlur(hsv[:, :, 1], (5, 5), 0)
        otsu_threshold, binary = cv2.threshold(saturation, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

        mask_u8 = (binary > 0).astype(np.uint8) * 255
        k_close = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (CLOSE_KERNEL, CLOSE_KERNEL))
        k_open = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (OPEN_KERNEL, OPEN_KERNEL))
        mask_u8 = cv2.morphologyEx(mask_u8, cv2.MORPH_CLOSE, k_close)
        mask_u8 = cv2.morphologyEx(mask_u8, cv2.MORPH_OPEN, k_open)

        mask = clean_mask(mask_u8 > 0)
        final_mask = mask.astype(np.uint8) * 255

        overlay = thumbnail.copy()
        area = final_mask > 0
        green = np.zeros_like(thumbnail)
        green[:, :, 1] = 255
        overlay[area] = (0.65 * overlay[area] + 0.35 * green[area]).astype(np.uint8)
        contours, _ = cv2.findContours(final_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        cv2.drawContours(overlay, contours, -1, (0, 255, 0), 2)

        return {"thumbnail": thumbnail, "mask": final_mask, "overlay": overlay,
                "otsu_threshold": float(otsu_threshold), "tissue_fraction": float(np.mean(mask)),
                "width": width, "height": height}
    finally:
        slide.close()
        gc.collect()


def combined_path(base):
    return OUTPUT_DIR / f"{base}_tissue_detection.png"


def mask_path(base):
    return OUTPUT_DIR / f"{base}_tissue_mask.png"


def write_png_atomic(path, arr):
    tmp = path.with_name(path.stem + ".tmp.png")
    if not cv2.imwrite(str(tmp), arr):
        raise IOError(f"gagal menulis {tmp}")
    os.replace(tmp, path)


def load_mask(base):
    """Dipakai tahap patch extraction: kembalikan mask bool (thumbnail resolution)."""
    return cv2.imread(str(mask_path(base)), cv2.IMREAD_GRAYSCALE) > 127


def process_slide(slide_path):
    """Dijalankan di proses worker. Mengembalikan 1 baris summary."""
    slide_path = Path(slide_path)
    base, t0 = slide_path.stem, time.time()
    try:
        r = tissue_detection_otsu(slide_path, MASK_DOWNSAMPLE)
        mask_rgb = cv2.cvtColor(r["mask"], cv2.COLOR_GRAY2RGB)
        combined = np.hstack([r["thumbnail"], mask_rgb, r["overlay"]])
        write_png_atomic(mask_path(base), r["mask"])                       # mask dulu
        write_png_atomic(combined_path(base), cv2.cvtColor(combined, cv2.COLOR_RGB2BGR))
        th, tw = r["thumbnail"].shape[:2]
        return {"slide": base, "status": "BERHASIL",
                "otsu_threshold": r["otsu_threshold"], "tissue_fraction": r["tissue_fraction"],
                "width": r["width"], "height": r["height"],
                "thumb_width": tw, "thumb_height": th,
                "scale_x": r["width"] / tw, "scale_y": r["height"] / th,
                "time_sec": time.time() - t0, "error": ""}
    except Exception as e:
        return {"slide": base, "status": "GAGAL_TISSUE_DETECTION",
                "time_sec": time.time() - t0, "error": repr(e)[:300]}


def outputs_exist(base):
    return mask_path(base).exists() and combined_path(base).exists()


def save_summary(rows):
    new = pd.DataFrame(rows.values())
    tmp = SUMMARY_CSV.with_suffix(".csv.tmp")
    new.to_csv(tmp, index=False)
    os.replace(tmp, SUMMARY_CSV)


def main():
    if not WSI_INPUT_DIR.exists():
        raise FileNotFoundError(f"Folder tidak ditemukan: {WSI_INPUT_DIR}")
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    for p in OUTPUT_DIR.glob("*.tmp.png"):          # sisa tulis yang terputus
        p.unlink()

    svs_files = sorted(WSI_INPUT_DIR.rglob("*.svs"))
    print(f"Ditemukan {len(svs_files)} WSI | worker: {N_WORKERS}")
    if not svs_files:
        raise RuntimeError("Tidak ada file .svs. Jalankan download_wsi_luad.py dulu.")

    rows = {}
    if SUMMARY_CSV.exists():
        try:
            rows = {r["slide"]: r for r in pd.read_csv(SUMMARY_CSV).to_dict("records")}
        except Exception:
            rows = {}

    todo = [p for p in svs_files if not outputs_exist(p.stem)]
    print(f"Sudah selesai: {len(svs_files) - len(todo)} | akan diproses: {len(todo)}")

    ok = fail = 0
    t_all = time.time()
    with ProcessPoolExecutor(N_WORKERS) as ex:
        futs = {ex.submit(process_slide, str(p)): p for p in todo}
        for i, f in enumerate(as_completed(futs), start=1):
            row = f.result()
            rows[row["slide"]] = row
            save_summary(rows)
            if row["status"] == "BERHASIL":
                ok += 1
                print(f"[{i}/{len(todo)}] OK  {row['slide']} | tissue {row['tissue_fraction']:.3f} "
                      f"| {row['time_sec']:.1f}s", flush=True)
            else:
                fail += 1
                print(f"[{i}/{len(todo)}] GAGAL {row['slide']}: {row.get('error')}", flush=True)

    print("\n" + "=" * 60)
    print(f"SELESAI dalam {time.time() - t_all:.0f} detik | berhasil {ok} | gagal {fail}")
    print(f"Hasil  : {OUTPUT_DIR}\nSummary: {SUMMARY_CSV}")
    if fail:
        print("Jalankan ulang skrip untuk mencoba slide yang gagal.")


if __name__ == "__main__":
    main()


# **Patch Extraction**

In [ ]:
import argparse
import gc
import os
import shutil
import time
import zipfile
from concurrent.futures import ProcessPoolExecutor, as_completed
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import openslide

cv2.setNumThreads(1)

# ---------------- LOKASI ----------------
BASE_DIR = Path(os.environ.get("TCGA_BASE", "/workspace/TCGA-LUAD"))
WSI_DIR = BASE_DIR / "WSI"
TISSUE_DIR = BASE_DIR / "Tissue_Detection"
SUMMARY_CSV = TISSUE_DIR / "summary_tissue_detection.csv"
SPLIT_CSV = WSI_DIR / "split.csv"
LABELS_CSV = WSI_DIR / "labels.csv"

PATCH_DIR = BASE_DIR / "Patches"
MANIFEST_DIR = PATCH_DIR / "manifests"
PATCH_SUMMARY_CSV = PATCH_DIR / "patch_summary.csv"
PATCH_MANIFEST_CSV = PATCH_DIR / "patch_manifest.csv"

# ---------------- PARAMETER ----------------
PATCH_SIZE = 224
STRIDE = 224
LEVEL = 0
MIN_TISSUE = 0.70
PRE_MASK_MIN = 0.50
BLUR_THRESHOLD = 100.0          # kalibrasi dulu dengan --calibrate

PATCH_FORMAT = "png"            # "png" (lossless) atau "jpg"
JPEG_QUALITY = 95
PNG_COMPRESSION = 3

N_WORKERS = int(os.environ.get("N_WORKERS", min(os.cpu_count() or 4, 16)))
MIN_FREE_GB = 30.0              # berhenti rapi jika sisa ruang volume di bawah ini

CALIBRATE_SLIDES = 3
CALIBRATE_SAMPLES = 300
SEED = 42


# ---------------- MASK & GRID ----------------
def load_mask(stem, info):
    tw, th = int(info["thumb_width"]), int(info["thumb_height"])
    mp = TISSUE_DIR / f"{stem}_tissue_mask.png"
    if mp.exists():
        mask = cv2.imread(str(mp), cv2.IMREAD_GRAYSCALE) > 127
    else:
        img = cv2.imread(str(TISSUE_DIR / f"{stem}_tissue_detection.png"), cv2.IMREAD_COLOR)
        if img is None:
            raise FileNotFoundError(f"hasil tissue detection tidak ada: {stem}")
        mask = img[:, tw:2 * tw, 0] > 127
    if mask.shape != (th, tw):
        raise ValueError(f"ukuran mask {mask.shape} != ({th}, {tw})")
    return mask


def candidate_grid(mask, width, height, sx, sy):
    xs = np.arange(0, width - PATCH_SIZE + 1, STRIDE)
    ys = np.arange(0, height - PATCH_SIZE + 1, STRIDE)
    gx, gy = np.meshgrid(xs, ys)
    gx, gy = gx.ravel(), gy.ravel()
    n_grid = len(gx)
    mh, mw = mask.shape
    x0 = np.clip(np.floor(gx / sx).astype(int), 0, mw)
    x1 = np.clip(np.ceil((gx + PATCH_SIZE) / sx).astype(int), 0, mw)
    y0 = np.clip(np.floor(gy / sy).astype(int), 0, mh)
    y1 = np.clip(np.ceil((gy + PATCH_SIZE) / sy).astype(int), 0, mh)
    integ = cv2.integral(mask.astype(np.uint8))
    s = integ[y1, x1] - integ[y0, x1] - integ[y1, x0] + integ[y0, x0]
    cov = s / np.maximum((x1 - x0) * (y1 - y0), 1)
    keep = cov >= PRE_MASK_MIN
    return gx[keep], gy[keep], cov[keep], n_grid


# ---------------- FILTER & ENCODE ----------------
def read_patch(slide, x, y):
    return np.array(slide.read_region((int(x), int(y)), LEVEL, (PATCH_SIZE, PATCH_SIZE)).convert("RGB"))


def tissue_fraction(rgb, sat_threshold):
    sat = cv2.GaussianBlur(cv2.cvtColor(rgb, cv2.COLOR_RGB2HSV)[:, :, 1], (5, 5), 0)
    return float(np.mean(sat > sat_threshold))


def blur_score(rgb):
    return float(cv2.Laplacian(cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY), cv2.CV_64F).var())


def encode_patch(rgb):
    bgr = cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR)
    if PATCH_FORMAT == "jpg":
        ok, buf = cv2.imencode(".jpg", bgr, [cv2.IMWRITE_JPEG_QUALITY, JPEG_QUALITY])
    else:
        ok, buf = cv2.imencode(".png", bgr, [cv2.IMWRITE_PNG_COMPRESSION, PNG_COMPRESSION])
    if not ok:
        raise IOError("gagal meng-encode patch")
    return buf.tobytes()


def sat_threshold_of(info):
    t = info.get("otsu_threshold", np.nan)
    return 20.0 if pd.isna(t) else float(t)


# ---------------- DAFTAR SLIDE ----------------
def load_slide_table():
    if SPLIT_CSV.exists():
        meta = pd.read_csv(SPLIT_CSV)
    elif LABELS_CSV.exists():
        meta = pd.read_csv(LABELS_CSV)
        meta["split"] = "all"
        print("PERINGATAN: split.csv tidak ada, memakai labels.csv (folder 'all').")
    else:
        raise FileNotFoundError("split.csv / labels.csv tidak ditemukan di folder WSI.")
    meta["slide"] = meta["file_name"].map(lambda n: Path(n).stem)

    summ = pd.read_csv(SUMMARY_CSV)
    summ = summ[summ["status"] == "BERHASIL"].set_index("slide")
    wsi_index = {p.stem: p for p in WSI_DIR.rglob("*.svs")}

    rows = []
    for _, r in meta.iterrows():
        s = r["slide"]
        if s in summ.index and s in wsi_index:
            rows.append((s, str(wsi_index[s]), summ.loc[s].to_dict(), r.to_dict()))
        else:
            print(f"  lewati {s}: belum ada di summary tissue detection atau WSI tidak ditemukan")
    return rows


# ---------------- KALIBRASI ----------------
def calibrate(rows):
    rng = np.random.default_rng(SEED)
    fracs, blurs, imgs = [], [], []
    for stem, wsi_path, info, _ in rows[:CALIBRATE_SLIDES]:
        mask = load_mask(stem, info)
        gx, gy, _, _ = candidate_grid(mask, int(info["width"]), int(info["height"]),
                                      float(info["scale_x"]), float(info["scale_y"]))
        idx = rng.permutation(len(gx))[:CALIBRATE_SAMPLES]
        T = sat_threshold_of(info)
        slide = openslide.OpenSlide(wsi_path)
        try:
            for i in idx:
                rgb = read_patch(slide, gx[i], gy[i])
                fracs.append(tissue_fraction(rgb, T))
                blurs.append(blur_score(rgb))
                imgs.append(rgb)
        finally:
            slide.close()
        print(f"  kalibrasi {stem}: {len(idx)} patch")

    fracs, blurs = np.array(fracs), np.array(blurs)
    ok = fracs >= MIN_TISSUE
    print(f"\nSampel patch: {len(fracs)}")
    print("Tissue fraction persentil 5/25/50/75:", np.percentile(fracs, [5, 25, 50, 75]).round(2))
    print(f"Lolos tissue >= {MIN_TISSUE:.0%}: {ok.mean() * 100:.1f}%")
    if ok.any():
        print("Blur score (lolos tissue) persentil 5/10/25/50:", np.percentile(blurs[ok], [5, 10, 25, 50]).round(1))
        print(f"BLUR_THRESHOLD = {BLUR_THRESHOLD} -> membuang {(blurs[ok] < BLUR_THRESHOLD).mean() * 100:.1f}% "
              "dari patch yang lolos tissue")
        order = np.argsort(np.where(ok, blurs, np.inf))[:12]
        tiles = []
        for i in order:
            t = cv2.cvtColor(imgs[i], cv2.COLOR_RGB2BGR).copy()
            cv2.putText(t, f"blur={blurs[i]:.0f}", (4, 16), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 1)
            tiles.append(t)
        while len(tiles) % 6:
            tiles.append(np.zeros_like(tiles[0]))
        grid = np.vstack([np.hstack(tiles[r:r + 6]) for r in range(0, len(tiles), 6)])
        PATCH_DIR.mkdir(parents=True, exist_ok=True)
        out = PATCH_DIR / "calibrasi_blur_terendah.png"
        cv2.imwrite(str(out), grid)
        print(f"Gambar 12 patch blur terendah: {out}  (unduh/lihat di JupyterLab)")
        print("Naikkan BLUR_THRESHOLD jika patch yang masih tajam ikut terbuang.")


# ---------------- EKSTRAKSI 1 SLIDE (dijalankan di worker) ----------------
def extract_slide(job):
    stem, wsi_path, info, meta = job
    t0 = time.time()
    split, label = str(meta.get("split", "all")), str(meta.get("label", "unknown"))
    dest_dir = PATCH_DIR / split / label
    dest, tmp = dest_dir / f"{stem}.zip", dest_dir / f"{stem}.zip.tmp"
    ext = "jpg" if PATCH_FORMAT == "jpg" else "png"
    try:
        if shutil.disk_usage(PATCH_DIR).free < MIN_FREE_GB * 1e9:
            return {"slide": stem, "status": "DISKFULL"}
        dest_dir.mkdir(parents=True, exist_ok=True)
        width, height = int(info["width"]), int(info["height"])
        T = sat_threshold_of(info)
        mask = load_mask(stem, info)
        gx, gy, cov, n_grid = candidate_grid(mask, width, height,
                                             float(info["scale_x"]), float(info["scale_y"]))
        manifest, rej_t, rej_b = [], 0, 0
        slide = openslide.OpenSlide(wsi_path)
        try:
            if tuple(slide.dimensions) != (width, height):
                raise ValueError(f"dimensi WSI {slide.dimensions} != summary {(width, height)}")
            with zipfile.ZipFile(tmp, "w", compression=zipfile.ZIP_STORED) as zf:
                for i in range(len(gx)):
                    x, y = int(gx[i]), int(gy[i])
                    rgb = read_patch(slide, x, y)
                    tf = tissue_fraction(rgb, T)
                    if tf < MIN_TISSUE:
                        rej_t += 1
                        continue
                    bs = blur_score(rgb)
                    if bs < BLUR_THRESHOLD:
                        rej_b += 1
                        continue
                    fname = f"{stem}_x{x}_y{y}.{ext}"
                    zf.writestr(f"{stem}/{fname}", encode_patch(rgb))
                    manifest.append({"patch": fname, "slide": stem,
                                     "patient_id": meta.get("patient_id", ""), "label": label, "split": split,
                                     "x": x, "y": y, "size": PATCH_SIZE, "level": LEVEL,
                                     "mask_coverage": round(float(cov[i]), 3),
                                     "tissue_fraction": round(tf, 3), "blur_score": round(bs, 1)})
                zf.writestr(f"{stem}/manifest.csv", pd.DataFrame(manifest).to_csv(index=False))
        finally:
            slide.close()
        size = tmp.stat().st_size
        os.replace(tmp, dest)
        mtmp = MANIFEST_DIR / f"{stem}.csv.tmp"
        pd.DataFrame(manifest).to_csv(mtmp, index=False)
        os.replace(mtmp, MANIFEST_DIR / f"{stem}.csv")          # penanda selesai (terakhir)
        return {"slide": stem, "status": "OK", "label": label, "split": split, "grid_cells": n_grid,
                "after_mask": len(gx), "rejected_tissue": rej_t, "rejected_blur": rej_b,
                "saved": len(manifest), "size_mb": round(size / 1e6, 1), "time_sec": round(time.time() - t0)}
    except Exception as e:
        Path(tmp).unlink(missing_ok=True)
        return {"slide": stem, "status": "GAGAL", "error": repr(e)[:300]}
    finally:
        gc.collect()


# ---------------- RINGKASAN ----------------
def save_patch_summary(stats):
    new = pd.DataFrame([stats])
    if PATCH_SUMMARY_CSV.exists():
        try:
            new = pd.concat([pd.read_csv(PATCH_SUMMARY_CSV), new], ignore_index=True)
        except Exception:
            pass
    new = new.drop_duplicates("slide", keep="last")
    tmp = PATCH_SUMMARY_CSV.with_suffix(".csv.tmp")
    new.to_csv(tmp, index=False)
    os.replace(tmp, PATCH_SUMMARY_CSV)


def merge_manifests():
    frames = [pd.read_csv(f) for f in sorted(MANIFEST_DIR.glob("*.csv")) if f.stat().st_size > 0]
    if frames:
        pd.concat(frames, ignore_index=True).to_csv(PATCH_MANIFEST_CSV, index=False)
        print(f"Manifest gabungan: {PATCH_MANIFEST_CSV}")


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--calibrate", action="store_true")
    args = ap.parse_args()

    PATCH_DIR.mkdir(parents=True, exist_ok=True)
    MANIFEST_DIR.mkdir(parents=True, exist_ok=True)
    for p in PATCH_DIR.rglob("*.zip.tmp"):
        p.unlink()

    rows = load_slide_table()
    print(f"Slide siap diproses: {len(rows)}")
    if not rows:
        raise SystemExit("Tidak ada slide. Pastikan tissue detection dan split sudah dijalankan.")
    if args.calibrate:
        calibrate(rows)
        return

    todo = [r for r in rows if not (MANIFEST_DIR / f"{r[0]}.csv").exists()]
    print("=" * 70)
    print(f"PATCH {PATCH_SIZE}x{PATCH_SIZE} | tissue >= {MIN_TISSUE:.0%} | blur >= {BLUR_THRESHOLD} | "
          f"{PATCH_FORMAT} | worker {N_WORKERS}")
    print(f"Sudah selesai: {len(rows) - len(todo)} | akan diproses: {len(todo)} | "
          f"ruang kosong {shutil.disk_usage(PATCH_DIR).free / 1e9:.0f} GB")
    print("=" * 70, flush=True)

    ok = fail = 0
    stopped = False
    ex = ProcessPoolExecutor(N_WORKERS)
    try:
        futs = [ex.submit(extract_slide, j) for j in todo]
        for n, f in enumerate(as_completed(futs), start=1):
            s = f.result()
            if s["status"] == "DISKFULL":
                print("\nBERHENTI: ruang volume hampir habis. Kosongkan/perbesar volume, lalu jalankan ulang.")
                stopped = True
                break
            if s["status"] == "OK":
                ok += 1
                save_patch_summary(s)
                print(f"[{n}/{len(todo)}] OK {s['slide']} | {s['saved']} patch ({s['size_mb']:.0f} MB) | "
                      f"dibuang tissue={s['rejected_tissue']} blur={s['rejected_blur']} | {s['time_sec']}s", flush=True)
            else:
                fail += 1
                print(f"[{n}/{len(todo)}] GAGAL {s['slide']}: {s.get('error')}", flush=True)
    except KeyboardInterrupt:
        print("\nDihentikan manual. Jalankan ulang untuk melanjutkan.")
        stopped = True
    finally:
        ex.shutdown(wait=not stopped, cancel_futures=True)
        merge_manifests()

    print("\n" + "=" * 70)
    print(f"SELESAI | berhasil {ok} | gagal {fail} | Patch: {PATCH_DIR}")
    print("=" * 70)


if __name__ == "__main__":
    main()


# **Macenko**

In [ ]:
import gc
import os
import shutil
import time
import zipfile
from concurrent.futures import ProcessPoolExecutor, as_completed
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")                   # headless
import matplotlib.pyplot as plt

cv2.setNumThreads(1)

# ---------------- LOKASI ----------------
BASE_DIR = Path(os.environ.get("TCGA_BASE", "/workspace/TCGA-LUAD"))
PATCH_DIR = BASE_DIR / "Patches"
OUT_DIR = BASE_DIR / "Patches_Macenko_paper"
SAMPLE_DIR = BASE_DIR / "samples_orig_paper"

REF_DIR = OUT_DIR / "reference"
REF_PARAMS = REF_DIR / "reference_params.npz"
REF_PNG = REF_DIR / "reference.png"
MANIFEST_DIR = OUT_DIR / "manifests"
VALID_DIR = OUT_DIR / "validation"
QC_DIR = OUT_DIR / "qc"
SUMMARY_CSV = OUT_DIR / "macenko_summary.csv"
MANIFEST_ALL = OUT_DIR / "macenko_manifest.csv"

# ---------------- PARAMETER ----------------
IO, ALPHA, BETA = 255.0, 1, 0.15   # IO=255: I dinormalisasi ke [0,1] (Eq.1); alpha & beta dari jurnal
STAIN_MAX_PCT = 99                 # Sec.4: persentil ke-99 sebagai pseudo-maksimum
MIN_TISSUE_PIXELS = 500
MIN_STAIN_PIXELS = 50              # min. piksel mayoritas per stain untuk menghitung pseudo-maksimum
FORMULA_TAG = "macenko2009_log10_svd_majority_v1"

FIT_LEVEL = os.environ.get("FIT_LEVEL", "patch")   # "patch" (per patch, fallback ke slide) atau "slide"
SLIDE_FIT_PATCHES = 30

REFERENCE_PATCH = None
USE_SAVED_REFERENCE = True
REF_SLIDES = 10
REF_PATCHES_PER_SLIDE = 10

OUT_FORMAT = "same"                 # "same", "png", atau "jpg"
JPEG_QUALITY = 95
PNG_COMPRESSION = 3

SAMPLE_N = 100                      # patch asli yang disimpan per slide
DELETE_ORIGINAL_AFTER_NORM = True   # hapus zip asli penuh setelah hasil terverifikasi

VALIDATE_N = 30
N_SHOW = 8
MAX_FAIL_FRAC = 0.01
MAX_MEDIAN_ANGLE = 10.0
MAX_P95_ANGLE = 20.0
MAXC_RATIO_RANGE = (0.85, 1.15)
MAX_BG_SHIFT = 0.05
MAX_CLIP = 0.02

N_WORKERS = int(os.environ.get("N_WORKERS", min(os.cpu_count() or 4, 16)))
MIN_FREE_GB = 30.0
SEED = 42
IMG_EXT = (".png", ".jpg", ".jpeg")


# ---------------- MACENKO (mengikuti jurnal) ----------------
def to_od(pix):
    """Eq.(1): OD = -log10(I), I di [0,1]. Nilai 0 di-clamp ke 1/255 agar log terdefinisi."""
    I = np.clip(pix.astype(np.float64), 1.0, 255.0) / IO
    return -np.log10(I)


def od_to_rgb(OD):
    """Kebalikan Eq.(1): I = IO * 10^(-OD)."""
    return IO * np.power(10.0, -OD)


def stain_pseudo_max(C, tissue):
    """Sec.4: untuk tiap stain, ambil piksel jaringan yang MAYORITAS stain itu (konsentrasinya
    lebih besar dari stain lain), lalu persentil ke-99 dari konsentrasinya sebagai pseudo-maksimum."""
    majority = np.argmax(C, axis=0)
    out = np.empty(C.shape[0])
    for k in range(C.shape[0]):
        vals = C[k, tissue & (majority == k)]
        if vals.size < MIN_STAIN_PIXELS:
            raise ValueError(f"piksel mayoritas stain {k} terlalu sedikit ({vals.size})")
        out[k] = np.percentile(vals, STAIN_MAX_PCT)
    return out


def fit_from_od(OD):
    """Algorithm 1 (SVD-geodesic) + Sec.4. Kembalikan (HE [3x2], pseudo-maksimum [2])."""
    tissue = ~np.any(OD < BETA, axis=1)                       # step 2: buang piksel tanpa stain (OD < beta)
    ODhat = OD[tissue]
    if ODhat.shape[0] < MIN_TISSUE_PIXELS:
        raise ValueError(f"piksel jaringan terlalu sedikit ({ODhat.shape[0]})")

    _, _, Vt = np.linalg.svd(ODhat, full_matrices=False)      # step 3: SVD pada tuple OD (tanpa centering)
    plane = Vt[:2].T.copy()                                   # step 4: bidang dari 2 singular value terbesar
    if plane[:, 0].sum() < 0:                                 # arah SVD pertama dibuat positif (OD >= 0)
        plane[:, 0] = -plane[:, 0]

    proj = ODhat @ plane                                      # step 5: proyeksi ke bidang
    proj = proj / np.maximum(np.linalg.norm(proj, axis=1, keepdims=True), 1e-12)   # ... normalisasi satuan
    phi = np.arctan2(proj[:, 1], proj[:, 0])                  # step 6: sudut thd arah SVD pertama
    lo, hi = np.percentile(phi, [ALPHA, 100 - ALPHA])         # step 7: ekstrem robust (alpha, 100-alpha)
    v1 = plane @ np.array([np.cos(lo), np.sin(lo)])           # step 8: kembali ke ruang OD
    v2 = plane @ np.array([np.cos(hi), np.sin(hi)])
    HE = np.stack([v1, v2], axis=1) if v1[0] > v2[0] else np.stack([v2, v1], axis=1)   # kol 0 = H (R lebih besar)

    C = np.linalg.pinv(HE) @ OD.T                             # Eq.(2): S = V^-1 OD
    maxC = stain_pseudo_max(C, tissue)                        # Sec.4
    if not (np.all(np.isfinite(HE)) and np.all(np.isfinite(maxC)) and np.all(maxC > 1e-6)):
        raise ValueError("estimasi stain tidak valid")
    return HE, maxC


def macenko_fit(pix):
    return fit_from_od(to_od(pix))


def macenko_normalize(rgb, HE_ref, maxC_ref, HE, maxC):
    """Pindahkan ke ruang stain referensi (Sec.3) dan samakan intensitas stain (Sec.4)."""
    h, w, _ = rgb.shape
    OD = to_od(rgb.reshape(-1, 3))
    C = np.linalg.pinv(HE) @ OD.T                             # Eq.(2)
    scale = maxC / maxC_ref
    if not np.all(np.isfinite(scale)) or np.any(scale < 1e-6):
        raise ValueError("skala konsentrasi tidak valid")
    out = od_to_rgb(HE_ref @ (C / scale[:, None]))            # OD = V_ref S, lalu I = IO*10^-OD
    return np.clip(np.rint(out), 0, 255).T.reshape(h, w, 3).astype(np.uint8)


# ---------------- BANTU ----------------
def load_rgb(path):
    return cv2.cvtColor(cv2.imread(str(path)), cv2.COLOR_BGR2RGB)


def decode_rgb(data):
    bgr = cv2.imdecode(np.frombuffer(data, np.uint8), cv2.IMREAD_COLOR)
    if bgr is None:
        raise IOError("gagal decode patch")
    return cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)


def encode_rgb(rgb, ext):
    bgr = cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR)
    if ext == ".jpg":
        ok, buf = cv2.imencode(".jpg", bgr, [cv2.IMWRITE_JPEG_QUALITY, JPEG_QUALITY])
    else:
        ok, buf = cv2.imencode(".png", bgr, [cv2.IMWRITE_PNG_COMPRESSION, PNG_COMPRESSION])
    if not ok:
        raise IOError("gagal encode patch")
    return buf.tobytes()


def out_ext(name):
    if OUT_FORMAT == "same":
        return ".jpg" if Path(name).suffix.lower() in (".jpg", ".jpeg") else ".png"
    return ".jpg" if OUT_FORMAT == "jpg" else ".png"


def image_names(zf):
    return [i.filename for i in zf.infolist() if Path(i.filename).suffix.lower() in IMG_EXT]


def tissue_mask(rgb):
    return ~np.any(to_od(rgb.reshape(-1, 3)) < BETA, axis=1)


def lab_mean(rgb, mask):
    lab = cv2.cvtColor(rgb.astype(np.float32) / 255.0, cv2.COLOR_RGB2LAB).reshape(-1, 3)
    return lab[mask].mean(axis=0)


def angle_deg(u, v):
    c = abs(float(u @ v)) / (np.linalg.norm(u) * np.linalg.norm(v))
    return float(np.degrees(np.arccos(np.clip(c, 0.0, 1.0))))


def white_frac(rgb):
    return float(np.mean(np.all(rgb >= 220, axis=2)))


def clip_frac(rgb, mask):
    px = rgb.reshape(-1, 3)[mask]
    return float(np.mean(np.any((px == 0) | (px == 255), axis=1))) if len(px) else 0.0


def list_zips():
    return sorted(PATCH_DIR.glob("*/*/*.zip"))


def split_label(zpath):
    rel = zpath.relative_to(PATCH_DIR).parts
    return rel[0], rel[1]


# ---------------- REFERENSI ----------------
def get_reference(zips):
    if REFERENCE_PATCH is not None:
        ref_rgb = load_rgb(REFERENCE_PATCH)
        HE, mC = macenko_fit(ref_rgb.reshape(-1, 3))
        src = f"REFERENCE_PATCH ({REFERENCE_PATCH})"
    elif USE_SAVED_REFERENCE and REF_PARAMS.exists() and REF_PNG.exists():
        z = np.load(REF_PARAMS)
        if "formula" not in z.files or str(z["formula"]) != FORMULA_TAG:
            raise SystemExit(f"{REF_PARAMS} dibuat dengan rumus lain. Hapus folder {REF_DIR} lalu jalankan ulang.")
        return z["HE"], z["maxC"], load_rgb(REF_PNG), "referensi tersimpan"
    else:
        if not zips:
            raise SystemExit("Tidak ada zip patch untuk memilih referensi.")
        rng = np.random.default_rng(SEED)
        train = [z for z in zips if split_label(z)[0] == "train"] or zips
        picked = [train[i] for i in rng.permutation(len(train))[:REF_SLIDES]]
        imgs, fits, srcs = [], [], []
        for zp in picked:
            with zipfile.ZipFile(zp) as zf:
                names = image_names(zf)
                for j in rng.permutation(len(names))[:REF_PATCHES_PER_SLIDE]:
                    im = decode_rgb(zf.read(names[j]))
                    try:
                        fit = macenko_fit(im.reshape(-1, 3))
                    except ValueError:
                        continue
                    imgs.append(im); fits.append(fit); srcs.append(f"{zp.stem}/{Path(names[j]).name}")
        if len(fits) < 3:
            raise SystemExit("Terlalu sedikit patch valid untuk memilih referensi.")
        maxCs = np.array([f[1] for f in fits])
        feat = np.hstack([np.array([f[0].ravel() for f in fits]), maxCs / np.median(maxCs, axis=0)])
        best = int(np.argmin(np.linalg.norm(feat - np.median(feat, axis=0), axis=1)))   # medoid
        ref_rgb, (HE, mC) = imgs[best], fits[best]
        src = f"medoid dari {len(fits)} patch train ({srcs[best]})"
    REF_DIR.mkdir(parents=True, exist_ok=True)
    np.savez(REF_PARAMS, HE=HE, maxC=mC, formula=FORMULA_TAG)
    cv2.imwrite(str(REF_PNG), cv2.cvtColor(ref_rgb, cv2.COLOR_RGB2BGR))
    return HE, mC, ref_rgb, src


# ---------------- VALIDASI ----------------
def validate_pairs(pairs, ref):
    HE_ref, maxC_ref, ref_lab = ref["HE"], ref["maxC"], ref["lab"]
    rows = []
    for name, o, n in pairs:
        m = tissue_mask(o)
        lab_o, lab_n = lab_mean(o, m), lab_mean(n, m)
        row = {"patch": name,
               "L_before": lab_o[0], "a_before": lab_o[1], "b_before": lab_o[2],
               "L_after": lab_n[0], "a_after": lab_n[1], "b_after": lab_n[2],
               "dE_ref_before": float(np.linalg.norm(lab_o - ref_lab)),
               "dE_ref_after": float(np.linalg.norm(lab_n - ref_lab)),
               "white_before": white_frac(o), "white_after": white_frac(n),
               "clip_after": clip_frac(n, m),
               "shape_ok": bool(n.shape == o.shape and n.dtype == np.uint8),
               "finite_ok": bool(np.all(np.isfinite(n)))}
        try:
            HE_n, maxC_n = macenko_fit(n.reshape(-1, 3))
            row["angle_H"] = angle_deg(HE_n[:, 0], HE_ref[:, 0])
            row["angle_E"] = angle_deg(HE_n[:, 1], HE_ref[:, 1])
            r = maxC_n / maxC_ref
            row["ratio_maxC_H"], row["ratio_maxC_E"] = float(r[0]), float(r[1])
        except ValueError:
            row["angle_H"] = row["angle_E"] = row["ratio_maxC_H"] = row["ratio_maxC_E"] = np.nan
        rows.append(row)
    return pd.DataFrame(rows)


def save_qc_slide(stem, pairs, ref_rgb):
    if not pairs:
        return
    idx = np.linspace(0, len(pairs) - 1, min(N_SHOW, len(pairs))).astype(int)
    fig, axes = plt.subplots(2, len(idx) + 1, figsize=(2.4 * (len(idx) + 1), 5.2))
    axes[0, 0].imshow(ref_rgb); axes[0, 0].set_title("REFERENSI", fontsize=9)
    for c, j in enumerate(idx, start=1):
        axes[0, c].imshow(pairs[j][1]); axes[0, c].set_title("asli", fontsize=9)
        axes[1, c].imshow(pairs[j][2]); axes[1, c].set_title("Macenko", fontsize=9)
    for a in axes.ravel():
        a.axis("off")
    plt.suptitle(f"{stem} - sebelum vs sesudah Macenko (mode {FIT_LEVEL})")
    plt.tight_layout()
    plt.savefig(QC_DIR / f"{stem}.png", dpi=90)
    plt.close(fig)


# ---------------- NORMALISASI 1 SLIDE ----------------
def normalize_slide(zin_path, tmp_out, stem, ref):
    HE_ref, maxC_ref = ref["HE"], ref["maxC"]
    rng = np.random.default_rng(SEED)
    stats = {"slide": stem, "patches_in": 0, "norm_patch": 0, "norm_slide_fallback": 0, "dropped": 0}
    kept, pairs = {}, []
    cache = {"params": None, "failed": False}

    with zipfile.ZipFile(zin_path) as zin:
        names = image_names(zin)
        stats["patches_in"] = len(names)
        manifest_name = f"{stem}/manifest.csv"
        manifest = pd.read_csv(zin.open(manifest_name)) if manifest_name in zin.namelist() else None
        val_set = {names[i] for i in rng.choice(len(names), min(VALIDATE_N, len(names)), replace=False)}

        def slide_params():
            if cache["failed"]:
                raise ValueError("estimasi stain level slide gagal")
            if cache["params"] is None:
                try:
                    pick = rng.permutation(len(names))[:SLIDE_FIT_PATCHES]
                    pix = np.concatenate([decode_rgb(zin.read(names[j])).reshape(-1, 3) for j in pick])
                    if len(pix) > 2_000_000:
                        pix = pix[rng.choice(len(pix), 2_000_000, replace=False)]
                    cache["params"] = macenko_fit(pix)
                except Exception:
                    cache["failed"] = True
                    raise
            return cache["params"]

        with zipfile.ZipFile(tmp_out, "w", compression=zipfile.ZIP_STORED) as zout:
            for name in names:
                try:
                    rgb = decode_rgb(zin.read(name))
                    if FIT_LEVEL == "slide":
                        HE, mC = slide_params(); mode = "slide"
                    else:
                        try:
                            HE, mC = macenko_fit(rgb.reshape(-1, 3)); mode = "patch"
                        except ValueError:
                            HE, mC = slide_params(); mode = "slide_fallback"
                    out = macenko_normalize(rgb, HE_ref, maxC_ref, HE, mC)
                except Exception:
                    stats["dropped"] += 1
                    continue
                base = Path(name).name
                new_base = Path(base).stem + out_ext(base)
                zout.writestr(f"{stem}/{new_base}", encode_rgb(out, out_ext(base)))
                kept[base] = (new_base, mode)
                stats["norm_patch" if mode in ("patch", "slide") else "norm_slide_fallback"] += 1
                if name in val_set:
                    pairs.append((new_base, rgb, out))

            if manifest is not None:
                manifest = manifest[manifest["patch"].isin(kept)].copy()
                manifest["macenko_mode"] = manifest["patch"].map(lambda p: kept[p][1])
                manifest["patch"] = manifest["patch"].map(lambda p: kept[p][0])
            else:
                manifest = pd.DataFrame({"patch": [v[0] for v in kept.values()],
                                         "macenko_mode": [v[1] for v in kept.values()]})
            zout.writestr(manifest_name, manifest.to_csv(index=False))
    return manifest, pairs, stats, kept


def write_orig_sample(zin_path, stem, kept):
    """Simpan SAMPLE_N patch asli (lokasi sama dengan hasil normalisasi) ke samples_orig_paper/<slide>.zip."""
    SAMPLE_DIR.mkdir(parents=True, exist_ok=True)
    dest, tmp = SAMPLE_DIR / f"{stem}.zip", SAMPLE_DIR / f"{stem}.zip.tmp"
    rng = np.random.default_rng(SEED)
    with zipfile.ZipFile(zin_path) as zin:
        full = {Path(n).name: n for n in image_names(zin)}
        bases = sorted(b for b in kept if b in full)
        pick = [bases[i] for i in rng.permutation(len(bases))[:SAMPLE_N]]
        with zipfile.ZipFile(tmp, "w", compression=zipfile.ZIP_STORED) as zout:
            for b in pick:
                zout.writestr(f"{stem}/{b}", zin.read(full[b]))
    os.replace(tmp, dest)
    return len(pick)


# ---------------- 1 SLIDE (dijalankan di worker) ----------------
def process_zip(job):
    zpath, ref = Path(job[0]), job[1]
    stem = zpath.stem
    split, label = split_label(zpath)
    dest = OUT_DIR / split / label / f"{stem}.zip"
    tmp = dest.with_name(dest.name + ".tmp")
    t0 = time.time()
    try:
        if shutil.disk_usage(OUT_DIR).free < MIN_FREE_GB * 1e9:
            return {"slide": stem, "status": "DISKFULL"}
        dest.parent.mkdir(parents=True, exist_ok=True)

        manifest, pairs, stats, kept = normalize_slide(zpath, tmp, stem, ref)
        n_ok = stats["norm_patch"] + stats["norm_slide_fallback"]
        if n_ok == 0:
            raise ValueError("0 patch berhasil dinormalisasi")

        with zipfile.ZipFile(tmp) as z:                     # verifikasi jumlah patch di zip hasil
            if len(image_names(z)) != n_ok:
                raise IOError("jumlah patch di zip hasil tidak cocok")

        stats["sample_orig"] = write_orig_sample(zpath, stem, kept)
        validate_pairs(pairs, ref).to_csv(VALID_DIR / f"{stem}.csv", index=False)
        save_qc_slide(stem, pairs, ref["rgb"])
        manifest.to_csv(MANIFEST_DIR / f"{stem}.csv", index=False)

        stats["size_mb"] = round(tmp.stat().st_size / 1e6, 1)
        os.replace(tmp, dest)                               # zip hasil ada = slide selesai
        if DELETE_ORIGINAL_AFTER_NORM:
            zpath.unlink(missing_ok=True)
        stats.update(status="OK", label=label, split=split, time_sec=round(time.time() - t0))
        return stats
    except Exception as e:
        tmp.unlink(missing_ok=True)
        return {"slide": stem, "status": "GAGAL", "error": repr(e)[:300]}
    finally:
        gc.collect()


def save_summary(stats):
    new = pd.DataFrame([stats])
    if SUMMARY_CSV.exists():
        try:
            new = pd.concat([pd.read_csv(SUMMARY_CSV), new], ignore_index=True)
        except Exception:
            pass
    new = new.drop_duplicates("slide", keep="last")
    tmp = SUMMARY_CSV.with_suffix(".csv.tmp")
    new.to_csv(tmp, index=False)
    os.replace(tmp, SUMMARY_CSV)


def merge_manifests():
    frames = [pd.read_csv(f) for f in sorted(MANIFEST_DIR.glob("*.csv")) if f.stat().st_size > 0]
    if frames:
        pd.concat(frames, ignore_index=True).to_csv(MANIFEST_ALL, index=False)
        print(f"Manifest gabungan: {MANIFEST_ALL}")


# ---------------- LAPORAN VALIDASI ----------------
def final_report(ref):
    files = sorted(VALID_DIR.glob("*.csv"))
    if not files or not SUMMARY_CSV.exists():
        print("Belum ada data validasi.")
        return
    rep = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)
    summ = pd.read_csv(SUMMARY_CSV)
    rep.to_csv(OUT_DIR / "macenko_report_all.csv", index=False)

    n_in, n_drop = int(summ["patches_in"].sum()), int(summ["dropped"].sum())
    fail_frac = n_drop / max(n_in, 1)
    fb_frac = summ["norm_slide_fallback"].sum() / max(n_in, 1)
    ang = np.concatenate([rep["angle_H"].dropna().values, rep["angle_E"].dropna().values])
    ratio = np.concatenate([rep["ratio_maxC_H"].dropna().values, rep["ratio_maxC_E"].dropna().values])
    std_b = rep[["L_before", "a_before", "b_before"]].std().sum()
    std_a = rep[["L_after", "a_after", "b_after"]].std().sum()
    bg_shift = float((rep["white_after"] - rep["white_before"]).abs().mean())

    checks = [
        ("1. Hampir semua patch berhasil dinormalisasi", fail_frac <= MAX_FAIL_FRAC,
         f"{n_in - n_drop}/{n_in} berhasil, dibuang {fail_frac * 100:.2f}% (batas {MAX_FAIL_FRAC * 100:.0f}%); "
         f"fallback ke stain slide: {fb_frac * 100:.2f}%"),
        ("2. Ukuran, tipe uint8, dan nilai valid (tanpa NaN)", bool(rep["shape_ok"].all() and rep["finite_ok"].all()),
         f"{len(rep)} patch divalidasi dari {len(files)} slide"),
        ("3. Arah vektor stain hasil ~ referensi",
         bool(len(ang) and np.median(ang) <= MAX_MEDIAN_ANGLE and np.percentile(ang, 95) <= MAX_P95_ANGLE),
         f"median {np.median(ang):.1f} deg, p95 {np.percentile(ang, 95):.1f} deg "
         f"(batas {MAX_MEDIAN_ANGLE:.0f}/{MAX_P95_ANGLE:.0f})" if len(ang) else "tidak bisa dihitung"),
        ("4. Intensitas stain hasil ~ referensi",
         bool(len(ratio) and MAXC_RATIO_RANGE[0] <= np.median(ratio) <= MAXC_RATIO_RANGE[1]),
         f"rasio konsentrasi maks median {np.median(ratio):.2f} (harus {MAXC_RATIO_RANGE[0]}-{MAXC_RATIO_RANGE[1]})"
         if len(ratio) else "tidak bisa dihitung"),
        ("5. Warna jaringan lebih dekat ke referensi", bool(rep["dE_ref_after"].median() < rep["dE_ref_before"].median()),
         f"jarak LAB ke referensi median {rep['dE_ref_before'].median():.1f} -> {rep['dE_ref_after'].median():.1f}"),
        ("6. Variasi warna antar patch/slide tidak memburuk", bool(std_a <= std_b * 1.10),
         f"total std LAB {std_b:.2f} -> {std_a:.2f}"),
        ("7. Latar putih tidak berubah", bool(bg_shift <= MAX_BG_SHIFT),
         f"rata-rata pergeseran porsi latar {bg_shift:.3f} (batas {MAX_BG_SHIFT})"),
        ("8. Tidak ada clipping berlebihan", bool(rep["clip_after"].mean() <= MAX_CLIP),
         f"porsi piksel jaringan 0/255: {rep['clip_after'].mean() * 100:.2f}% (batas {MAX_CLIP * 100:.0f}%)"),
    ]
    print("\n" + "=" * 78)
    print("HASIL VALIDASI MACENKO (SELURUH SLIDE)")
    print("=" * 78)
    for name, ok, info in checks:
        print(f"[{'LULUS' if ok else 'PERIKSA'}] {name}\n          {info}")
    n_ok = sum(ok for _, ok, _ in checks)
    print("-" * 78)
    print(f"Kesimpulan: {n_ok}/{len(checks)} pemeriksaan lulus"
          + (" -> hasil normalisasi dapat dipakai." if n_ok == len(checks)
             else " -> lihat butir PERIKSA dan gambar QC sebelum melanjutkan."))

    fig, ax = plt.subplots(1, 2, figsize=(11, 4.5))
    ax[0].scatter(rep["a_before"], rep["b_before"], s=12, label="sebelum", alpha=0.5)
    ax[0].scatter(rep["a_after"], rep["b_after"], s=12, label="sesudah", alpha=0.5)
    ax[0].scatter([ref["lab"][1]], [ref["lab"][2]], marker="*", s=220, c="red", label="referensi")
    ax[0].set_xlabel("a*"); ax[0].set_ylabel("b*")
    ax[0].set_title("Warna jaringan per patch (LAB): harus mengumpul"); ax[0].legend()
    ax[1].hist(rep["dE_ref_before"], bins=25, alpha=0.7, label="sebelum")
    ax[1].hist(rep["dE_ref_after"], bins=25, alpha=0.7, label="sesudah")
    ax[1].set_xlabel("jarak warna ke referensi (LAB)"); ax[1].set_title("Harus bergeser ke kiri"); ax[1].legend()
    plt.tight_layout()
    plt.savefig(OUT_DIR / "qc_sebaran_warna.png", dpi=110)
    plt.close(fig)
    print(f"Laporan: {OUT_DIR / 'macenko_report_all.csv'} | grafik: {OUT_DIR / 'qc_sebaran_warna.png'}")


# ---------------- MAIN ----------------
def main():
    for d in (OUT_DIR, REF_DIR, MANIFEST_DIR, VALID_DIR, QC_DIR, SAMPLE_DIR):
        d.mkdir(parents=True, exist_ok=True)
    for p in list(OUT_DIR.rglob("*.zip.tmp")) + list(SAMPLE_DIR.glob("*.zip.tmp")):
        p.unlink()

    zips = list_zips()
    print(f"Zip patch asli tersisa: {len(zips)}")
    HE_ref, maxC_ref, ref_rgb, src = get_reference(zips)
    ref = {"HE": HE_ref, "maxC": maxC_ref, "rgb": ref_rgb, "lab": lab_mean(ref_rgb, tissue_mask(ref_rgb))}
    print(f"Referensi : {src}")

    todo = []
    for zp in zips:
        split, label = split_label(zp)
        dest = OUT_DIR / split / label / f"{zp.stem}.zip"
        if dest.exists():                                   # selesai; bereskan zip asli yang tertinggal
            if DELETE_ORIGINAL_AFTER_NORM:
                zp.unlink(missing_ok=True)
        else:
            todo.append(zp)
    print("=" * 70)
    print(f"MACENKO | mode {FIT_LEVEL} | worker {N_WORKERS} | akan diproses: {len(todo)} | "
          f"hapus asli setelah selesai: {DELETE_ORIGINAL_AFTER_NORM}")
    print(f"Ruang kosong: {shutil.disk_usage(OUT_DIR).free / 1e9:.0f} GB")
    print("=" * 70, flush=True)

    ok = fail = 0
    stopped = False
    ex = ProcessPoolExecutor(N_WORKERS)
    try:
        futs = [ex.submit(process_zip, (str(zp), ref)) for zp in todo]
        for n, f in enumerate(as_completed(futs), start=1):
            s = f.result()
            if s["status"] == "DISKFULL":
                print("\nBERHENTI: ruang volume hampir habis. Kosongkan/perbesar volume, lalu jalankan ulang.")
                stopped = True
                break
            if s["status"] == "OK":
                ok += 1
                save_summary(s)
                print(f"[{n}/{len(todo)}] OK {s['slide']} | {s['norm_patch'] + s['norm_slide_fallback']}/"
                      f"{s['patches_in']} patch ({s['size_mb']:.0f} MB) | fallback {s['norm_slide_fallback']} | "
                      f"dibuang {s['dropped']} | {s['time_sec']}s", flush=True)
            else:
                fail += 1
                print(f"[{n}/{len(todo)}] GAGAL {s['slide']}: {s.get('error')}", flush=True)
    except KeyboardInterrupt:
        print("\nDihentikan manual. Jalankan ulang untuk melanjutkan.")
        stopped = True
    finally:
        ex.shutdown(wait=not stopped, cancel_futures=True)
        merge_manifests()

    print("\n" + "=" * 70)
    print(f"SELESAI | berhasil {ok} | gagal {fail} | Hasil: {OUT_DIR}")
    print("=" * 70)
    final_report(ref)


if __name__ == "__main__":
    main()
